In [1]:
import pandas as pd
import numpy as np
import datetime

import sys
sys.path.append("/Users/derekdewald/Documents/Python/Github_Repo/d_py_functions")



In [2]:
from daily_etl_folder_mgmt import generate_objects_automated_py

In [5]:
import os
os.listdir('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/')

['.DS_Store',
 'knowledge_base.xlsx',
 'object_manual_published.xlsx',
 'consolidated_datset.xlsx']

In [6]:
knowledge_base_df = pd.read_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/consolidated_datset.xlsx')

## Manual Test 

In [11]:
consolidated_notes = pd.concat([notes_df,manual_object_df.drop('Order',axis=1)])

# Take all Processes from Notes, so we can extract Definitions
process_list = consolidated_notes['Process'].unique().tolist()
process_list.extend(consolidated_notes[consolidated_notes['Categorization']=='Process Step']['Word'].unique().tolist())
process_list = list(set(process_list))

# Extract Definitions to include into Consolidated Notes.
notes_from_def = definition_df[definition_df['Process'].isin(process_list)][['Process','Categorization','Word','Definition']]

# Create a Consolidated notes list (Before Incorporating Definitions).

final_df = pd.concat([
    consolidated_notes,
    notes_from_def])

In [13]:
process_list

['Model Interpretability',
 'Feasibility Assessment',
 'Desired State Metrics',
 'Machine Learning Lifecycle',
 'Option Generation',
 'Monitoring',
 'Optimization',
 'Documentation and Knowledge Sharing',
 'Feature Selection',
 'Implementation Planning',
 'Model Evaluation',
 'Feature Engineering',
 'Solution Design',
 'Behavioural Economics',
 'Data Dictionary',
 'Root Cause Analysis',
 'Data Collection',
 'Testing and Validation',
 'Current State Analysis',
 'Best Linear Unbiased Estimator',
 'Execution',
 'Bias, Fairness, and Ethics',
 'Evaluation',
 'Historical Overview',
 'Documentation',
 'Root Cause Assessment',
 'Continuous Improvement',
 'Validation',
 'Governance, Risk, and Ethics',
 'Hyperparameter Tuning',
 'Training',
 'Notes',
 'Baseline Metrics (Current State)',
 'Current State Assessment',
 'Research and Exploration',
 'Problem Definition',
 'Prioritization and Decision Making',
 'Model Selection',
 'Exploratory Data Analysis',
 'Deployment',
 'Data Preparation',
 'Goal

In [6]:
consolidated_notes = pd.concat([notes_df,manual_object_df.drop('Order',axis=1)])

# Take all Processes from Notes, so we can extract Definitions
process_list = consolidated_notes['Process'].unique().tolist()

# Extract Definitions to include into Consolidated Notes.
notes_from_def = definition_df[definition_df['Process'].isin(process_list)][['Process','Categorization','Word','Definition']]

# Create a Consolidated notes list (Before Incorporating Definitions).

final_df = pd.concat([
    consolidated_notes,
    notes_from_def])

# Merge in Sub Processes
word_list = final_df['Word'].unique().tolist()
sub_processes = final_df[(final_df['Process'].isin(word_list))]
#sub_processes = sub_processes.drop(['Categorization'],axis=1).rename(columns={'Word':"Categorization",'Process':'Word','Definition':"Definition"})
# Include Process.

sub_processes['Definition'] = sub_processes['Word'].fillna("") + ": " + sub_processes['Definition'].fillna('')
sub_processes['Word'] = sub_processes['Process']
sub_processes.drop('Process',axis=1,inplace=True)
sub_processes = sub_processes.merge(final_df[['Word','Process']],on='Word',how='left')

final_df = pd.concat([
    final_df,
    sub_processes])

final_df = final_df.merge(definition_df[['Word','Definition']].rename(columns={'Definition':'Definition1'}),on='Word',how='left')
final_df['Definition'] = np.where(final_df['Definition'].notnull(),final_df['Definition'],final_df['Definition1'])
final_df.drop('Definition1',axis=1,inplace=True)
#final_df.reset_index(inplace=True)
#final_df.rename(columns={'index':'default_order'},inplace=True)

# Sorting
# Process - Has to be Alphabetical. Do not actually Care Order.
# Categorization - Based on Manually defined Order, imported from Objects_Manual.
# Words Matter when they are part of a process, generally when not then alphabetically preferred. 

# Merge in Process  Filter
proc_filter = manual_object_df[(manual_object_df['Process']=='Notes')&(manual_object_df['Categorization']=='Filter Order')][['Word','Order']].reset_index(drop=True).rename(columns={'Word':"Categorization",'Order':"proc_order"})
final_df = final_df.merge(proc_filter,on='Categorization',how='left')

# Merge in Word Order.
final_df = final_df.merge(manual_object_df.drop('Categorization',axis=1).rename(columns={'Order':'word_order'}),on=['Process','Word'],how='left')

final_df.sort_values(['Process','word_order','proc_order','Word'],inplace=True)
final_df.drop(['word_order','proc_order'],axis=1,inplace=True)